# Importing libraries

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("dair-ai/emotion", "split")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,1
15998,i feel like this was such a rude comment and i...,3


# Dataset preprocessing

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [4]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        # For multiclass, output_dim = number_of_classes
        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        # If bidirectional=True, hidden vectors are doubled in size
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1) Embedding lookup
        embedded = self.embedding(input_ids)
        # 2) LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3) Extract final hidden state
        if self.lstm.bidirectional:
            # concatenate forward & backward final hidden states
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4) Dropout
        hidden = self.dropout(hidden)

        # 5) Fully connected layer -> logits of shape [batch_size, output_dim]
        output = self.fc(hidden)
        return output

# Instancing the LSTM model, criterion and optimizer

In [6]:
embedding_dim = 128
hidden_dim = 128
output_dim = test_df['label'].nunique()
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [8]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Forward pass -> logits: [batch_size, num_classes]
        logits = model(input_ids)
        # CrossEntropyLoss expects [batch_size, num_classes] vs. [batch_size] labels
        loss = criterion(logits, labels)

        # Backprop and optimize
        loss.backward()
        optimizer.step()

        # Track loss
        losses.append(loss.item())

        # Convert logits -> predicted classes
        preds_cls = torch.argmax(logits, dim=1)

        # Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # Accumulate predictions and labels for metric calculations
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate average loss and overall accuracy
    avg_loss = sum(losses) / len(losses)
    accuracy = float(correct_predictions) / len(data_loader.dataset)

    # Calculate macro metrics
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Calculate per-class F1-scores
    # This will return a NumPy array of length = num_classes
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class

def eval_model(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds_cls = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    avg_loss = sum(losses) / len(losses)
    accuracy = float(correct_predictions) / len(data_loader.dataset)

    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)

    return accuracy, avg_loss, precision, recall, f1_macro, f1_per_class


# Training loop

In [9]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class = train_epoch(
            model, train_loader, optimizer, criterion, device
        )
        
        val_acc, val_loss, val_prec, val_rec, val_f1_macro, val_f1_per_class = eval_model(
            model, val_loader, criterion, device
        )
        
        # Print macro stats
        print(f"Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, "
              f"Precision(macro): {train_prec:.4f}, Recall(macro): {train_rec:.4f}, "
              f"F1(macro): {train_f1_macro:.4f}")
        
        # Print per-class F1 for train
        print(f"F1 Per Class (Train): {train_f1_per_class}")
        
        print(f"Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, "
              f"Precision(macro): {val_prec:.4f}, Recall(macro): {val_rec:.4f}, "
              f"F1(macro): {val_f1_macro:.4f}")
        
        # Print per-class F1 for val
        print(f"F1 Per Class (Val):   {val_f1_per_class}")
        print("--------------------------------------------------")
    
    # Return the final metrics from the last epoch, if you like
    return (
        train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class,
        val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class
    )


In [10]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=[
    'seed', 
    'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1', 'train_f1_per_class',
    'val_loss',   'val_acc',   'val_prec',   'val_rec',   'val_f1',   'val_f1_per_class',
    'test_loss',  'test_acc',  'test_prec',  'test_rec',  'test_f1',  'test_f1_per_class',
    'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
    'max_memory_usage_test',  'max_vram_usage_test',  'total_time_test'
])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1_macro, train_f1_per_class, val_acc,   val_loss,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1, test_f1_per_class = retval

    new_row = pd.DataFrame([[
        seed,
        train_loss, train_acc, train_prec, train_rec, train_f1_macro, train_f1_per_class,
        val_loss,   val_acc,   val_prec,   val_rec,   val_f1_macro,   val_f1_per_class,
        test_loss,  test_acc,  test_prec,  test_rec,  test_f1,  test_f1_per_class,
        max_memory_usage_train, max_vram_usage_train, total_time_train,
        max_memory_usage_test,  max_vram_usage_test,  total_time_test
    ]], columns=results.columns)

    results = pd.concat([results, new_row], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 1.3722, Accuracy: 0.4798, Precision(macro): 0.3188, Recall(macro): 0.2797, F1(macro): 0.2481
F1 Per Class (Train): [0.53385918 0.60412939 0.         0.0154335  0.33176944 0.00343053]
Val Loss: 1.0441, Accuracy: 0.6260, Precision(macro): 0.4102, Recall(macro): 0.4151, F1(macro): 0.3844
F1 Per Class (Val):   [0.72876712 0.72924579 0.         0.23121387 0.61706783 0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.7166, Accuracy: 0.7497, Precision(macro): 0.6750, Recall(macro): 0.5708, F1(macro): 0.5801
F1 Per Class (Train): [0.84243112 0.82581741 0.29520697 0.6121939  0.7446703  0.16012085]
Val Loss: 0.4970, Accuracy: 0.8385, Precision(macro): 0.7990, Recall(macro): 0.7585, F1(macro): 0.7654
F1 Per Class (Val):   [0.90874882 0.89139633 0.62763466 0.82289803 0.80487805 0.53658537]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.3348, Accuracy: 0.8860, Precision(macro): 0.8386, Recall(macro): 0.8168, F1(ma

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_21944\2112283703.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_row], ignore_index=True)
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 1.3485, Accuracy: 0.4764, Precision(macro): 0.3944, Recall(macro): 0.2915, F1(macro): 0.2790
F1 Per Class (Train): [0.52170587 0.59329796 0.00305577 0.23392975 0.31838721 0.00344234]
Val Loss: 0.8965, Accuracy: 0.6940, Precision(macro): 0.4510, Recall(macro): 0.4936, F1(macro): 0.4688
F1 Per Class (Val):   [0.79679144 0.78325123 0.         0.59966499 0.63316583 0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.6242, Accuracy: 0.7816, Precision(macro): 0.6955, Recall(macro): 0.6230, F1(macro): 0.6323
F1 Per Class (Train): [0.86049213 0.8486981  0.41433481 0.73175224 0.7504902  0.18776671]
Val Loss: 0.4423, Accuracy: 0.8445, Precision(macro): 0.8489, Recall(macro): 0.7242, F1(macro): 0.7402
F1 Per Class (Val):   [0.90080429 0.89589041 0.68902439 0.84719536 0.76150628 0.34693878]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.2943, Accuracy: 0.8966, Precision(macro): 0.8544, Recall(macro): 0.8490, F1(ma

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 1.3439, Accuracy: 0.4739, Precision(macro): 0.3463, Recall(macro): 0.2865, F1(macro): 0.2746
F1 Per Class (Train): [0.5355483  0.58431566 0.03023758 0.2292517  0.2683274  0.        ]
Val Loss: 0.8271, Accuracy: 0.7045, Precision(macro): 0.5735, Recall(macro): 0.5400, F1(macro): 0.5384
F1 Per Class (Val):   [0.80817052 0.77742549 0.44303797 0.58823529 0.61382114 0.        ]
--------------------------------------------------
Epoch 2/5
Train Loss: 0.5777, Accuracy: 0.8024, Precision(macro): 0.7513, Recall(macro): 0.6749, F1(macro): 0.6938
F1 Per Class (Train): [0.87848523 0.85138676 0.54303534 0.77828489 0.74962293 0.3622251 ]
Val Loss: 0.4489, Accuracy: 0.8430, Precision(macro): 0.8226, Recall(macro): 0.7480, F1(macro): 0.7688
F1 Per Class (Val):   [0.90940767 0.88289567 0.63050847 0.83044983 0.77404922 0.58536585]
--------------------------------------------------
Epoch 3/5
Train Loss: 0.2693, Accuracy: 0.9060, Precision(macro): 0.8742, Recall(macro): 0.8604, F1(ma

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [11]:
results.to_csv('results/lstm_multiclass3.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,train_f1_per_class,val_loss,val_acc,val_prec,...,test_prec,test_rec,test_f1,test_f1_per_class,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.131170,0.947438,0.919540,0.919627,0.919515,"[0.9765046668812359, 0.9645496211766906, 0.884...",0.260038,0.9075,0.867344,...,0.845575,0.869858,0.856109,"[0.9449618966977138, 0.9220489977728286, 0.757...",1185.890625,231.648438,28.987977,1185.921875,194.691406,0.890015
1,3,0.123203,0.952063,0.927071,0.927608,0.927327,"[0.9766244906712417, 0.9678684849617037, 0.897...",0.254530,0.9100,0.879807,...,0.833212,0.841912,0.836996,"[0.9348392701998263, 0.9146608315098468, 0.755...",1186.195312,232.634766,30.469750,1186.191406,195.162109,1.001851
2,5,0.118201,0.951125,0.927267,0.924641,0.925904,"[0.9776111408677022, 0.9674166744468303, 0.900...",0.285798,0.9035,0.885155,...,0.861389,0.840342,0.848748,"[0.9427083333333334, 0.9217993079584775, 0.762...",1201.121094,232.513672,30.508418,1201.121094,194.691406,0.989571


In [12]:
torch.save(model.state_dict(), 'results/lstm_multiclass3.pth')